In [1]:
from collections import defaultdict

def read_corpus(filename):
    with open(filename, "r") as file:
        return file.read().splitlines()

def preprocess(words):
    cleaned = []
    for word in words:
        word = word.lower().strip()
        if word:
            cleaned.append(word)
    return cleaned

def build_vocab(words):
    vocab = defaultdict(int)

    for word in words:
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] += 1

    return vocab

def get_pair_frequencies(vocab):

    pairs = defaultdict(int)

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])

            pairs[pair] += freq

    return pairs


def get_best_pair(pair_freq):

    return max(pair_freq, key=pair_freq.get)

def merge_pair(pair, vocab):

    merged_vocab = {}

    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word in vocab:

        new_word = word.replace(bigram, replacement)

        merged_vocab[new_word] = vocab[word]

    return merged_vocab

def train_bpe(vocab, num_merges):

    merges = []

    print("\n========== TRAINING ==========\n")

    for i in range(num_merges):

        pair_freq = get_pair_frequencies(vocab)

        if len(pair_freq) == 0:
            break

        best_pair = get_best_pair(pair_freq)

        print("Merge", i+1)
        print("Best Pair :", best_pair)
        print("Frequency :", pair_freq[best_pair])
        print()

        merges.append(best_pair)

        vocab = merge_pair(best_pair, vocab)

    return vocab, merges


def build_final_vocab(vocab):

    tokens = set()

    for word in vocab:

        for token in word.split():
            tokens.add(token)

    return sorted(tokens)


def encode(word, merges):

    tokens = list(word)
    tokens.append("</w>")

    for pair in merges:

        i = 0

        while i < len(tokens)-1:

            if tokens[i] == pair[0] and tokens[i+1] == pair[1]:

                tokens[i] = pair[0] + pair[1]
                del tokens[i+1]

            else:
                i += 1

    return tokens


def decode(tokens):

    word = "".join(tokens)

    word = word.replace("</w>", "")

    return word

def main():

    print("="*60)
    print("CUSTOM BYTE PAIR ENCODING (BPE) TOKENIZER")
    print("="*60)

    words = read_corpus("corpus.txt")

    print("\nSTEP 2: Corpus")
    print(words)

    words = preprocess(words)

    print("\nSTEP 3: Preprocessed Corpus")
    print(words)

    vocab = build_vocab(words)

    print("\nSTEP 4 & 5: Initial Vocabulary")

    for word, freq in vocab.items():
        print(word, ":", freq)

    vocab, merges = train_bpe(vocab, num_merges=10)

    final_vocab = build_final_vocab(vocab)

    print("\n========== FINAL VOCABULARY ==========")

    for token in final_vocab:
        print(token)

    print("\nVocabulary Size =", len(final_vocab))

    print("\n========== MERGE RULES ==========")

    for i, rule in enumerate(merges, 1):
        print(i, ":", rule)

    # Step 16
    test_words = [
        "lowest",
        "newer",
        "widest",
        "lower",
        "new",
        "low"
    ]

    print("\n========== ENCODING & DECODING ==========\n")

    for word in test_words:

        encoded = encode(word, merges)

        decoded = decode(encoded)

        print("Original :", word)
        print("Encoded  :", encoded)
        print("Decoded  :", decoded)
        print()

if __name__ == "__main__":
    main()

CUSTOM BYTE PAIR ENCODING (BPE) TOKENIZER

STEP 2: Corpus
['low', 'lower', 'lowest', 'new', 'newer', 'newest']

STEP 3: Preprocessed Corpus
['low', 'lower', 'lowest', 'new', 'newer', 'newest']

STEP 4 & 5: Initial Vocabulary
l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
n e w e s t </w> : 1

========== TRAINING ==========

Merge 1
Best Pair : ('w', 'e')
Frequency : 4

Merge 2
Best Pair : ('l', 'o')
Frequency : 3

Merge 3
Best Pair : ('n', 'e')
Frequency : 3

Merge 4
Best Pair : ('w', '</w>')
Frequency : 2

Merge 5
Best Pair : ('lo', 'we')
Frequency : 2

Merge 6
Best Pair : ('r', '</w>')
Frequency : 2

Merge 7
Best Pair : ('s', 't')
Frequency : 2

Merge 8
Best Pair : ('st', '</w>')
Frequency : 2

Merge 9
Best Pair : ('ne', 'we')
Frequency : 2

Merge 10
Best Pair : ('lo', 'w</w>')
Frequency : 1


========== FINAL VOCABULARY ==========
low</w>
lowe
ne
newe
r</w>
st</w>
w</w>

Vocabulary Size = 7

========== MERGE RULES ==========
1 : ('w', 'e')
2